# Упражнение: Quality gate в CI/CD: пропустить модель или остановить конвейер

## Задание

CI-конвейер обучил шесть моделей-кандидатов `r1…r6`. Все они (и текущая продакшен-модель `prod`)
должны оцениваться на **одной и той же** зафиксированной валидационной выборке из двух сегментов.

- `input.csv` — матрицы ошибок по сегментам: `run_id, segment, tp, fp, fn, tn`;
- `meta.csv` — результаты остальных шагов: `smoke_test` (`pass`/`fail`), `latency_p95_ms`, `size_mb`.

Посчитайте F1 и примените quality gate. Для каждого кандидата:

1. **F1 по сегменту** = 2·TP / (2·TP + FP + FN). **Общий F1** — по суммарным TP, FP, FN обоих сегментов.
2. Бейзлайн — константная модель «всегда 1» на той же выборке `prod`: TP = P, FP = N, FN = 0,
   где P = TP + FN и N = FP + TN, суммарно по сегментам.
3. Проверки (в таком порядке и с такими именами):
   - `smoke` — `smoke_test == 'pass'`;
   - `same_data` — в **каждом** сегменте у кандидата те же P и N, что у `prod`;
   - `vs_prod` — общий F1 ≥ F1(`prod`) + 0.005;
   - `vs_baseline` — общий F1 ≥ F1(бейзлайна) + 0.02;
   - `segments` — в каждом сегменте F1 ≥ F1 этого сегмента у `prod` − 0.01;
   - `latency` — `latency_p95_ms` ≤ 120;
   - `size` — `size_mb` ≤ 50.
4. `decision` = `PASS`, если пройдены все проверки, иначе `FAIL`; `failed` — имена непройденных проверок
   через `;` в порядке списка выше (или `none`).
5. `promote` = 1 у **одного** прошедшего кандидата с наибольшим общим F1, у остальных 0.

## Подсказки

- Удобно сначала развернуть таблицу: `pivot` по сегментам или `groupby('run_id')[['tp','fp','fn']].sum()` для общего F1.
- Проверка `same_data` ловит кандидата, оценённого на другой выборке: его метрика несравнима с `prod`,
  даже если выглядит хорошо.
- Соберите проверки в словарь `{имя: bool}` (Python сохраняет порядок ключей) — тогда
  `failed = ';'.join(k for k, ok in checks.items() if not ok) or 'none'`.

## Ожидаемый формат выходного файла

CSV с колонками `run_id, f1, delta_vs_prod, decision, failed, promote` — 6 строк в порядке `r1…r6`;
`delta_vs_prod` = F1 кандидата − F1(`prod`); `f1` и `delta_vs_prod` округлить до 4 знаков.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/cicd_ml
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.